In [1]:
import os
import pandas as pd
import commons as c
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Merge and save DFs for equiv, normal and balanced

In [2]:
def read_and_merge_csv_files(folder_path):
    # List to hold individual DataFrames
    dataframes = []

    # Loop through all files in the folder
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            file_path = os.path.join(folder_path, filename)
            # Read the CSV file into a DataFrame
            df = pd.read_csv(file_path)
            df = df.loc[~df['Name'].str.contains("Replace", na=False)]
            # Append the DataFrame to the list
            dataframes.append(df)

    # Concatenate all DataFrames in the list into a single DataFrame
    merged_df = pd.concat(dataframes, ignore_index=True)

    return merged_df

# Function to split the name column and create new columns
def split_name_column(name):
    name = name.replace('.qasm', '')
    parts = name.split('_')
    position = int(parts[5].replace('P', ''))
    qubit = parts[6].replace('Q', '')
    
    if len(parts) > 7: 
        parameters = parts[7].strip('[]') 
    else: 
        parameters = None

    return parts[0], parts[1], parts[3], parts[4], position, qubit, parameters

def get_gate_type(gate):
    single_qubit_gates = ["x", "h", "p", "t", "s", "z", "y", "id", "rx", "ry", "rz", "sx", "u", "u1", "u2", "u3"]
    multi_qubit_gates = ["swap", "rzz", "rxx", "cx", "cz", "cp", "ccx", "cswap", "ch"]
    if gate in single_qubit_gates:
        return 'Single_qubit'
    elif gate in multi_qubit_gates:
        return 'Multi_qubit'
    else:
        return 'Gate_not_supported'
    
# Function to categorize position based on percentage
def categorize_position(percentage):
    if percentage <= 20:
        return 'beginning'
    elif percentage <= 40:
        return 'pre_middle'
    elif percentage <= 60:
        return 'middle'
    elif percentage <= 80:
        return 'post_middle'
    else:
        return 'end'

In [3]:
# Function to split the name column and create new columns
def split_name_column_origin(name):
    name = name.replace('.qasm', '')
    parts = name.split('_')
    return parts[0], parts[3]

In [4]:
def get_dataframe_origin(model):
    """Generates a processed DataFrame for a given noise model, mutant type, and threshold."""

    # Get all the results in a df
    folder_path = f'./results_TEST/results_{model}_origin'
    #folder_path = f'./results_{model}/results_{mutant}_{threshold}'
    df = read_and_merge_csv_files(folder_path)
    
    # Categorize input type
    df['Input_type'] = df['Input'].str.split('_').str[0]

    # Split 'Name' column into multiple columns
    df[['Algorithm', 'Qubits_number']] = df['Name'].apply(lambda x: pd.Series(split_name_column_origin(x)))

    # Drop intermediate columns
    df = df.drop(columns=['Name'])

    # Map output types
    df['Output_type'] = df['Algorithm'].map(c.output_type)

    return df

In [5]:
def get_dataframe(model, mutant, df_char):
    """Generates a processed DataFrame for a given noise model, mutant type, and threshold."""

    # Get all the results in a df
    folder_path = f'./results_TEST/results_{model}_{mutant}'
    #folder_path = f'./results_{model}/results_{mutant}_{threshold}'
    df = read_and_merge_csv_files(folder_path)
    
    # Categorize input type
    df['Input_type'] = df['Input'].str.split('_').str[0]

    # Split 'Name' column into multiple columns
    df[['Algorithm', 'Qubits_number', 'Operator', 'Gate', 'Position', 'Qubits', 'Params']] = df['Name'].apply(lambda x: pd.Series(split_name_column(x)))

    # Categorize gate type
    df['Gate_type'] = df['Gate'].apply(get_gate_type)

    # Calculate position percentage and categorize
    df['max_position'] = df.groupby(['Algorithm', 'Qubits_number'])['Position'].transform('max')
    df['position_percentage'] = (df['Position'] / df['max_position']) * 100
    df['Relative_position'] = df['position_percentage'].apply(categorize_position)

    # Drop intermediate columns
    df = df.drop(columns=['max_position', 'position_percentage', 'Name'])

    # Merge with characteristics DataFrame
    merged_df = pd.merge(df_char, df, left_on=['qubits', 'algo'], right_on=['Qubits_number', 'Algorithm'], how='right')
    merged_df = merged_df.drop(columns=['qubits', 'algo'])

    # Map output types
    merged_df['Output_type'] = merged_df['Algorithm'].map(c.output_type)

    return merged_df

In [13]:
def process_characteristics(file_path):
    """Processes the characteristics Excel file into a DataFrame."""
    df_charac = pd.read_excel(file_path, usecols=[0, 2, 3, 5, 6, 7])
    df_charac['algo'] = df_charac.iloc[:, 0].str.split('_').str[0]
    df_charac['qubits'] = df_charac['qubits'].astype(str)
    return df_charac.drop(columns=[df_charac.columns[0]])

In [16]:
def process_metrics(complete_df):
    """Processes metrics and saves results to CSV."""

    selected_columns = complete_df[['Algorithm', 'Qubits_number', 'hardware', 'nature']]   
    new_rows = []

    for metric, metric_name in c.metrics.items():
        metric_df = selected_columns.copy()
        metric_df['metric'] = metric
        metric_df['ideal_distance'] = complete_df[f'Ideal_{metric_name}']
        metric_df['noisy_distance'] = complete_df[f'Noisy_{metric_name}']
        new_rows.append(metric_df)

    final_df = pd.concat(new_rows, ignore_index=True)
    # final_df = final_df.astype(c.type_dict)
    return final_df

In [17]:
xlsx_path = 'data/origin_qc/programs_characteristics.xlsx'
df_charac = process_characteristics(xlsx_path)
os.makedirs('results_TEST/dataframes/', exist_ok=True)
all_dfs = []

for hw in c.hardware:
    for m in ['equiv', 'normal']:
        df = get_dataframe(hw, m, df_charac)
        df['hardware'] = hw
        if m == 'equiv':
            df['nature'] = f'equivalent mutant' 
        else:
            df['nature'] = f'non-equivalent mutant' 
        all_dfs.append(df)
    
    df_origin = get_dataframe_origin(hw)
    df_origin['hardware'] = hw
    df_origin['nature'] = 'original program'  
    all_dfs.append(df_origin)

final_df = pd.concat(all_dfs, ignore_index=True)
final_df = process_metrics(final_df)
csv_path = 'results_TEST/dataframes/all_data.csv'
final_df.to_csv(csv_path, mode='w', header=True, index=False)